# Tomahawk 16 T: increased-pressure and shear proxy screen

Attach the **same original** `Tomahawk_16T_Full_MGRID_MHD_Power_Input.zip` as one Kaggle dataset, then choose **Save Version → Save & Run All**. Kaggle may mount it as a ZIP or extracted files. The input manifest and every member are checked before any calculation.

This notebook compares the original CSV pressure with the density increases needed for (a) **6.1 GWth per core at 46% efficiency and 3.5 GWe recirculation** and (b) **22 GWe at 40% efficiency and 3.0 GWe recirculation**. It evaluates the original q target and a candidate with q(0)=1.2, q(0.96)=4.0 and a linear coefficient of 1.0, on 121², 161² and 201² grids. The bundled fixed-boundary correction is adapted from 14 T to 16 T in memory; the CSV flux coordinate is converted using rho=sqrt(normalized_flux). No original package files are changed.

**Meaning of PASS:** a pass is limited to the bundled local Suydam and ballooning *proxies*. Its field solve prescribes the plasma current and uses a circular boundary. It does not prove that the higher-pressure plasma is stable in the 16-period 3-D device. The revised q target also changes modeled plasma current, whose signed drive and engineering feasibility must be checked separately.

Results: `/kaggle/working/Tomahawk_16T_Pressure_Stability_Screen_Results.zip`, with JSON, CSV, pressure profiles and q-profile diagnostics. Electric power is conditional arithmetic using the stated capture, efficiency and load assumptions. No 15.08 GWe anchor offset is used.


In [ ]:
import importlib
import subprocess
import sys
for package in ("numpy", "scipy"):
    try:
        importlib.import_module(package)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
import numpy as np
import scipy
print("Python", sys.version.split()[0], "NumPy", np.__version__, "SciPy", scipy.__version__)


In [ ]:
"""Reproducible 16 T pressure and shear *proxy* screen for Kaggle.

Uses the manifest-verified original source package. The bundled fixed-boundary
correction is adapted in memory to 16 T and to psi=rho**2. Its prescribed
current and circular boundary do not establish 3-D plasma stability.
"""

from __future__ import annotations

import csv
import hashlib
import importlib.util
import io
import json
import math
import os
import shutil
import tempfile
import zipfile
from pathlib import Path

import numpy as np


EXPECTED_MANIFEST = "797b100de7e11ec2a3af4d29ee25415a46951772bcc6f4ab5bb141e0a556a2a7"
PROFILE = "MHD/mhd_profile_sweep.csv"
SOLVER = "MHD/Tomahawk_MHD_FixedBoundary_Correction.py"
REFERENCE = "MHD/bosch_hale_uw_table6.csv"
MU0 = 4e-7 * math.pi
KEV_TO_J = 1.602176634e-16
FUSION_J = 17.6e6 * 1.602176634e-19
FUSION_VOLUME_M3 = 950.0
USABLE = (14.1 * 0.95 + 3.5 * 0.90) / 17.6
GRID_SIZES = (121, 161, 201)
Q_TARGETS = {
    "original_q": {"q0": 1.2, "q_at_rho_0p96": 5.808, "linear": 2.0},
    "revised_shear_candidate": {"q0": 1.2, "q_at_rho_0p96": 4.0, "linear": 1.0},
}


def sha(data):
    return hashlib.sha256(data).hexdigest()


def find_input():
    configured = os.environ.get("TOMAHAWK_SOURCE_ZIP")
    if configured:
        path = Path(configured)
        if not path.exists():
            raise FileNotFoundError(f"TOMAHAWK_SOURCE_ZIP not found: {path}")
        return path
    for root in (Path("/kaggle/input"), Path.cwd()):
        if not root.is_dir():
            continue
        for manifest in sorted(root.rglob("MANIFEST_SHA256.json")):
            if sha(manifest.read_bytes()) == EXPECTED_MANIFEST:
                return manifest.parent
        for archive in sorted(root.rglob("*.zip")):
            try:
                with zipfile.ZipFile(archive) as z:
                    if any(sha(z.read(n)) == EXPECTED_MANIFEST for n in z.namelist()
                           if n.endswith("MANIFEST_SHA256.json")):
                        return archive
            except (OSError, zipfile.BadZipFile):
                pass
    mounted = Path("/kaggle/input")
    available = sorted(p.name for p in mounted.iterdir()) if mounted.is_dir() else []
    raise FileNotFoundError(f"Original 16 T package missing. Kaggle datasets: {available}")


def load_source(path):
    if path.is_dir():
        reader = lambda name: (path / name).read_bytes()
        closer = lambda: None
    else:
        z = zipfile.ZipFile(path)
        names = [n for n in z.namelist() if n.endswith("MANIFEST_SHA256.json")]
        matches = [n for n in names if sha(z.read(n)) == EXPECTED_MANIFEST]
        if len(matches) != 1:
            z.close()
            raise ValueError("Expected source manifest is absent or duplicated")
        prefix = matches[0].removesuffix("MANIFEST_SHA256.json")
        reader = lambda name: z.read(prefix + name)
        closer = z.close
    try:
        manifest_bytes = reader("MANIFEST_SHA256.json")
        if sha(manifest_bytes) != EXPECTED_MANIFEST:
            raise ValueError("Manifest does not match the finished MGRID run")
        manifest = json.loads(manifest_bytes)
        if not all(member in manifest for member in (PROFILE, SOLVER, REFERENCE)):
            raise ValueError("Source package lacks profile, reference, or solver")
        for member, digest in manifest.items():
            if sha(reader(member)) != digest:
                raise ValueError(f"Missing or changed source member: {member}")
        return {key: reader(key) for key in (PROFILE, SOLVER, REFERENCE)}, manifest
    finally:
        closer()


def dt_reactivity(t_keV):
    t = np.asarray(t_keV, dtype=float)
    if not np.isfinite(t).all() or np.any(t < 0.2) or np.any(t > 100):
        raise ValueError("Bosch-Hale fit temperature is outside [0.2,100] keV")
    bg, mrc2 = 34.3827, 1.124656e6
    c1, c2, c3, c4 = 1.17302e-9, 1.51361e-2, 7.51886e-2, 4.60643e-3
    c5, c6, c7 = 1.35000e-2, -1.06750e-4, 1.36600e-5
    theta = t / (1 - t * (c2 + t * (c4 + t * c6)) /
                 (1 + t * (c3 + t * (c5 + t * c7))))
    xi = (bg**2 / (4 * theta)) ** (1 / 3)
    return 1e-6 * c1 * theta * np.sqrt(xi / (mrc2 * t**3)) * np.exp(-3 * xi)


def original_fusion_source(rows):
    psi = np.array([float(r["normalized_flux"]) for r in rows])
    n = np.array([float(r["density_m3"]) for r in rows])
    ti = np.array([float(r["ion_temperature_keV"]) for r in rows])
    te = np.array([float(r["electron_temperature_keV"]) for r in rows])
    p = np.array([float(r["pressure_Pa"]) for r in rows])
    if (not np.isclose(psi[0], 0) or not np.isclose(psi[-1], 1)
            or not np.all(np.diff(psi) > 0) or np.any(n <= 0) or np.any(p <= 0)):
        raise ValueError("Invalid source profile")
    pressure_error = float(np.max(np.abs(n * (ti + te) * KEV_TO_J / p - 1)))
    if pressure_error > 1e-4:
        raise ValueError("CSV pressure and total-ion density disagree")
    x, w = np.polynomial.legendre.leggauss(64)
    rho, weights = (x + 1) / 2, w / 2
    n_i = np.interp(rho**2, psi, n)
    ti_i = np.interp(rho**2, psi, ti)
    fusion_gw = float(FUSION_VOLUME_M3 * np.sum(weights * 2 * rho *
                      (n_i / 2)**2 * dt_reactivity(ti_i) * FUSION_J) / 1e9)
    return fusion_gw, pressure_error


def load_16t_proxy(module_path):
    spec = importlib.util.spec_from_file_location("tomahawk_verified_fixed_boundary", module_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    if module.B0_T != 14.0:
        raise ValueError("Unexpected bundled correction version")
    module.B0_T = 16.0
    module.F_TOROIDAL = module.R0_M * module.B0_T
    original_loader = module.load_pressure_profile

    def load_radius_mapped_profile(path):
        psi, pressure = original_loader(path)
        return np.sqrt(psi), pressure

    module.load_pressure_profile = load_radius_mapped_profile
    return module


def main():
    path = find_input()
    source, manifest = load_source(path)
    rows = list(csv.DictReader(io.StringIO(source[PROFILE].decode("utf-8-sig"))))
    reference = list(csv.DictReader(io.StringIO(source[REFERENCE].decode("utf-8-sig"))))
    xref = np.array([float(r["T_keV"]) for r in reference])
    yref = np.array([float(r["reactivity_cm3_s"]) for r in reference]) * 1e-6
    ref_error = float(np.max(np.abs(dt_reactivity(xref) / yref - 1)))
    if ref_error >= 0.005:
        raise ValueError("Reactivity fit does not agree with source table")
    source_gw, pressure_error = original_fusion_source(rows)
    target_40pct = (22 + 3) / (10 * USABLE * 0.4)
    scenarios = {
        "original_pressure": {"fusion_GWth_per_core": source_gw, "eta": 0.40, "recirc_GWe": 3.0},
        "combined_6p1": {"fusion_GWth_per_core": 6.1, "eta": 0.46, "recirc_GWe": 3.5},
        "source_only_22": {"fusion_GWth_per_core": target_40pct, "eta": 0.40, "recirc_GWe": 3.0},
    }
    output = Path(os.environ.get("TOMAHAWK_PRESSURE_OUTPUT_DIR", "/kaggle/working" if
                      Path("/kaggle/working").is_dir() else str(Path.cwd() / "pressure_screen_output")))
    output.mkdir(parents=True, exist_ok=True)
    summary = []
    with tempfile.TemporaryDirectory(prefix="tomahawk_pressure_") as temporary:
        work = Path(temporary)
        verified_script = work / "Tomahawk_MHD_FixedBoundary_Correction.py"
        verified_script.write_bytes(source[SOLVER])
        module = load_16t_proxy(verified_script)
        for label, case in scenarios.items():
            scale = math.sqrt(case["fusion_GWth_per_core"] / source_gw)
            case["density_pressure_multiplier"] = scale
            case["central_species_density_m3"] = float(rows[0]["density_m3"]) * scale / 2
            case["central_pressure_Pa"] = float(rows[0]["pressure_Pa"]) * scale
            case["vacuum_axis_beta_approx"] = 2 * MU0 * case["central_pressure_Pa"] / 16.0**2
            case["conditional_net_GWe_no_anchor_offset"] = (10 * case["fusion_GWth_per_core"] *
                                                            USABLE * case["eta"] - case["recirc_GWe"])
            case["pressure_profile"] = f"{label}_profile.csv"
            profile_path = output / case["pressure_profile"]
            with profile_path.open("w", newline="", encoding="utf-8") as stream:
                writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
                writer.writeheader()
                for original in rows:
                    entry = dict(original)
                    entry["density_m3"] = format(float(entry["density_m3"]) * scale, ".16g")
                    entry["pressure_Pa"] = format(float(entry["pressure_Pa"]) * scale, ".16g")
                    writer.writerow(entry)
            run_input = work / label / "input" / "MHD" / "mhd_profile_sweep.csv"
            run_input.parent.mkdir(parents=True)
            shutil.copyfile(profile_path, run_input)
            for target_name, q in Q_TARGETS.items():
                module.Q0_TARGET = q["q0"]
                module.Q_EDGE_TARGET = q["q_at_rho_0p96"]
                module.Q_LINEAR_TARGET = q["linear"]
                for grid in GRID_SIZES:
                    run_output = work / label / target_name / str(grid)
                    report = module.build_correction(run_input.parents[1], run_output, grid, grid)
                    metrics, gates = report["fixed_boundary_candidate"], report["gates"]
                    summary.append({
                        "scenario": label, "q_target": target_name, "grid_nr_nz": grid,
                        "fusion_GWth_per_core": case["fusion_GWth_per_core"],
                        "central_pressure_Pa": case["central_pressure_Pa"],
                        "q0_target": q["q0"], "q_rho0p96_target": q["q_at_rho_0p96"],
                        "q_linear_coefficient": q["linear"],
                        "q_at_rho_0p96_solved": metrics["q_at_rho_0_96"],
                        "max_q_target_relative_error": metrics["max_abs_q_target_relative_error"],
                        "plasma_current_model_MA": metrics["I_p_solved_MA"],
                        "GS_discrete_residual": metrics["gs_residual_relative"],
                        "divB_relative": metrics["divB_relative"],
                        "min_suydam_proxy_D": metrics["min_suydam_D"],
                        "max_ballooning_alpha_over_crit": metrics["max_ballooning_alpha_over_alpha_crit"],
                        "all_fixed_boundary_proxy_gates_pass": gates["fixed_boundary_correction_screen_pass"],
                        "proxy_gate_details": {key: value for key, value in gates.items()
                                               if key not in ("free_boundary_3D_MHD_pass", "qualified_design_pass")},
                    })
                    if grid == max(GRID_SIZES) and label == "combined_6p1":
                        shutil.copyfile(run_output / "MHD" / "fixed_boundary_q_profile.csv",
                                        output / f"combined_6p1_{target_name}_q_profile.csv")

    report = {
        "status": "PRESSURE_SHEAR_PROXY_SCREEN_NOT_STABILITY_PROOF",
        "input_source": str(path),
        "source_manifest_sha256": EXPECTED_MANIFEST,
        "bundled_proxy_sha256": manifest[SOLVER],
        "source_pressure_identity_max_relative_error": pressure_error,
        "source_bosch_hale_table_max_relative_error": ref_error,
        "coordinate_mapping": "source normalized_flux = rho^2; square root before bundled proxy uses rho",
        "adaptation": "bundled 14 T circular fixed-boundary proxy B0 and F_TOROIDAL set to 16 T in memory",
        "q_target_family": "q(rho)=q0+linear*rho+[(q96-q0-0.96*linear)/0.96^4]*rho^4",
        "q_targets": Q_TARGETS,
        "scenarios": scenarios,
        "mesh_grid_sizes": GRID_SIZES,
        "runs": summary,
        "lower_capture_combined_conditional_net_GWe": (
            10 * 6.1 * 0.46 * (14.1 * 0.90 + 3.5 * 0.80) / 17.6 - 3.5
        ),
        "limits": [
            "Pressure is evaluated in local Suydam and s-alpha proxies; it is not coupled into the prescribed-current field solve.",
            "The solver uses a circular fixed boundary, not the 16 T CAD elongation and 16-period free boundary.",
            "The revised q target requires about 26.4 MA in this model; signed current and engineering feasibility are unverified.",
            "A proxy pass is not a nonlinear or 3-D equilibrium stability result.",
            "95% neutron capture, 90% alpha deposition, 46% conversion efficiency and 3.5 GWe load are assumptions.",
        ],
    }
    json_file = output / "Tomahawk_16T_Pressure_Stability_Screen.json"
    csv_file = output / "Tomahawk_16T_Pressure_Stability_Screen.csv"
    json_file.write_text(json.dumps(report, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    columns = [k for k in summary[0] if k != "proxy_gate_details"]
    with csv_file.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=columns)
        writer.writeheader()
        writer.writerows({k: v for k, v in row.items() if k in columns} for row in summary)
    zip_file = output / "Tomahawk_16T_Pressure_Stability_Screen_Results.zip"
    files = [json_file, csv_file] + [output / c["pressure_profile"] for c in scenarios.values()]
    files += [output / f"combined_6p1_{target_name}_q_profile.csv" for target_name in Q_TARGETS]
    with zipfile.ZipFile(zip_file, "w", compression=zipfile.ZIP_DEFLATED) as z:
        for p in files:
            z.write(p, p.name)
    for row in summary:
        if row["scenario"] in ("original_pressure", "combined_6p1") and row["grid_nr_nz"] == 201:
            print(row["scenario"], row["q_target"], "ballooning ratio",
                  round(row["max_ballooning_alpha_over_crit"], 3), "proxy pass",
                  row["all_fixed_boundary_proxy_gates_pass"])
    print("Source:", path)
    print("PROXY SCREEN ONLY; 3-D stability is unproven.")
    print("DONE", zip_file)
    return report


if __name__ == "__main__":
    main()
